# Test Stage 13 — Is the input healthy?

Three questions about the signal arriving at the converter, answered from one set of captures
because they are really the same question: **can I trust what I am being given?**

| | Why it matters |
|---|---|
| Are the levels sane? | An overloaded input produces a distorted shape that still looks vaguely like a resonance — this project's signature failure |
| What else is in the band? | A clock harmonic near the coil frequency would be found by Stage 15's search and fitted as a spectacular high-Q resonance |
| Do the three measurement paths agree? | The register voltmeter, the memory capture and the averaged spectrum each have their own scaling and their own chance of a stray factor of two |

### Why this comes before looking for the coil

Searching a band you have not surveyed is how a stray signal becomes a discovery. Run this
first, then Stage 15.

### The gap this fills

**There is no input-overload detection anywhere in this design.** The output stage limits, and
the emulated resonator has a sticky overload flag — which is what has quietly been doing this
job for ten stages. Disable the emulator for real work and that flag goes with it, exactly when
an unknown coil, amplifier and gain chain start presenting unknown levels. Until a hardware
flag exists, this notebook is the guard.

---

### Simulation or the real trap?

**Run it twice.** Once now on the bench, which establishes the method and catalogues the
board's own stray signals. Again once the coil and amplifier are attached, which is the only
way to learn the real levels. Same notebook both times — only `SOURCE_NOTE` below changes.

## What you need

| | |
|---|---|
| Stages 1–7 | Passing |
| Input | Terminated for the survey; the real source connected for the level check |
| Time | About three minutes |

## Setup — drivers, capture and the spectrum

In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Environment shims. This board carries an older numpy and matplotlib than a
#  current desktop, and two conveniences used below arrived later than what is
#  installed here: nan_to_num's `nan=` keyword needs numpy >= 1.17, and
#  Axes.set_facecolor needs matplotlib >= 2.0. Provided once, rather than as
#  version tests scattered through the analysis cells.
# =============================================================================
def nan_fill(a, value):
    """np.nan_to_num(a, nan=value), for numpy versions without the keyword."""
    out = np.array(a, dtype=float, copy=True)
    out[~np.isfinite(out)] = value
    return out


def set_bg(ax, colour):
    """Axes background colour, across the matplotlib 1.x -> 2.x rename."""
    if hasattr(ax, 'set_facecolor'):
        ax.set_facecolor(colour)
    else:
        ax.set_axis_bgcolor(colour)


# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15, oshift=7)
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    COEF_FS = 16384.0                 # coefficient full scale -> gain LSB
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
    def gain_lsb(self, phi_deg, f=F0):
        """Smallest change in A this actuator can express at this phase. The
        two coefficients quantise independently, so the step in |A| is set by
        the LARGER of the two projections. Step 5c compares this against the
        width of the band a software loop would need to steer within."""
        phi = np.radians(phi_deg - 360.0 * self.LATENCY * f / FS_CLK)
        return 1.0 / (self.COEF_FS * max(abs(np.cos(phi)), abs(np.sin(phi))))
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready  (numpy %s, matplotlib %s)'
      % (np.__version__, matplotlib.__version__))

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    # WAIT FOR THE WRITER; DO NOT GUESS AT IT.
    #
    # This used to be one sleep of "as long as the record should take, plus
    # 5 ms", after which the writer was disarmed and memory read. Two things
    # were wrong with that, and both showed up on 2026-08-04 as an intermittent
    # 'unwritten words in the record' on captures that were perfectly healthy:
    #
    #  1. DMA_COUNT (0x34) was not consulted until AFTER the decision to stop
    #     had already been made. The sleep was a guess with no verification.
    #  2. DMA_COUNT reports samples HANDED TO the memory writer, not samples
    #     that have landed in DDR. The writer bursts, so the last burst can
    #     still be in flight when the count already reads complete.
    #
    # A fixed 5 ms of margin is also the wrong shape: it is 30% of a short
    # record and 0.1% of a long one, so the failure is rare, load-dependent and
    # looks like a hardware fault. It is not - the record was read a few
    # hundred microseconds early.
    #
    # Poll until the record is complete, then let the last burst land.
    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5         # generous, not 5 ms
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                                  # disarm
    time.sleep(0.002)                               # last burst reaches DDR

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    def _read_region(nbytes):
        fd = mm = None
        try:
            fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
            mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
            return mm[0:nbytes]
        finally:
            if mm is not None:
                mm.close()
            if fd is not None:
                os.close(fd)

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    _limit = max(8, 0.001 * u32.size)
    if got and int(np.count_nonzero(u32 == POISON)) > _limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms; if it was
        # merely late it will not. Retrying once separates the two, which the
        # single-shot check could not: it reported a transport fault for both.
        time.sleep(0.02)
        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        n_bad = int(np.count_nonzero(u32 == POISON))
        if n_bad > _limit:
            raise RuntimeError(
                'unwritten words in the record - transport fault, see Stage 6 '
                '(%d of %d words still unwritten 20 ms after the writer '
                'reported the record complete)' % (n_bad, u32.size))
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# ============================ CONFIGURATION ==================================
EMULATION  = True    # True: digital resonator, known answers, PASS gates.
                     # False: the real coil - same scan, gates become measurements

TANK_SH    = 3       # tank (Johnson) noise level, 6 dB per step
OUT_SH     = 4       # amplifier-floor level (off during sweeps automatically)

# The gain axis must BRACKET the self-oscillation boundary, or there is no
# boundary in the atlas and every stability margin the designer prints is
# meaningless. K rises ~0.73 per unit A, so onset is near A = 1.4 (+3 dB).
GAINS_DB   = np.arange(-26.0, 6.01, 2.0)     # feedback gain axis (dB of A)
PHIS       = np.arange(0.0, 360.0, 15.0)     # feedback phase axis: 24 points
SWEEP_PTS  = 15      # frequencies per mini-sweep at each grid point
SWEEP_SPAN = 2.5     # mini-sweep half-width in open-loop linewidths
SWEEP_N    = 2048    # samples per frequency point (halved at deep decimations)
SETTLE     = 0.02

# A point is OSCILLATING if, with the noise drive removed, it still holds this
# fraction of its driven power. A ratio, so it needs no absolute floor - on a
# digital plant the undriven floor is exactly zero.
SUS_OSC    = 0.15
DRIVE_STEP = 2       # noise-shift steps (12 dB) used to test loop linearity
# Grid cost: 17 x 24 points x ~1 s. About 7 minutes.
# =============================================================================

def coherence(z):
    """1 for pure noise, 0 for a pure tone; equals 1 - f^2 where f is the
    power fraction carried by a coherent tone. Reported as a diagnostic.
    Not used to classify points: for narrowband noise the effective sample
    count is small and this scatters ~30%, far too coarse to detect anything
    subtle."""
    p = np.abs(z) ** 2
    m = float(np.mean(p))
    return float(np.var(p)) / max(m * m, 1e-300)


def tone_fraction(R):
    return float(np.sqrt(max(0.0, 1.0 - R)))

---
## Step 1 — Are the levels sane?

Two numbers from a raw record tell you whether the front end is linear.

**How close to full scale.** Count samples within a few percent of the converter's limit. Any
at all is a warning; many is a measurement you should throw away.

**Peak-to-average ratio.** Thermal noise has a calculable one — for a complex record of N
samples the largest magnitude sits near `sqrt(ln N)` times the average. A clipped signal cannot
reach it, because clipping removes exactly the peaks that set it. So a low ratio says the
signal is being squashed even when nothing has quite hit the rail.

Headroom is reported in decibels. The gate below refuses anything with less than `MIN_HEADROOM_DB`.

In [ ]:
SOURCE_NOTE     = 'bench: input terminated'   # change when the coil is attached
# Full scale AT THE POINT capture() READS, which is the CIC output - not the
# converter.
#
# FIXED 2026-08-03. This was float(1 << 13), the 14-bit converter's full scale,
# compared against numbers that had been through a mixer and a decimator with
# about 96 dB of gain between them. With a TERMINATED input it reported -13.7 dB
# of headroom and 5268 of 8192 samples "near full scale", and told us to reduce
# the gain ahead of a converter that had nothing connected to it.
#
# The record contradicted itself, which is what gave it away: the same 8192
# samples had a crest factor of 3.23 against the 3.00 expected of pure noise -
# healthy, unclipped noise. A signal with 64% of its samples at full scale has a
# crest factor of about 1. Both cannot be true. The scale reference was wrong,
# not the input.
#
# The correct value follows from the chain, all of it stated in the RTL headers:
#
#   ADC full-scale sine            peak 2^13        (14-bit signed)
#   local oscillator               peak 2^15        (DWL = 16)
#   raw product                    peak 2^28
#   mixer, >> SHIFT = 6            peak 2^22        (red_pitaya_acq_mixer.v)
#     of which the baseband term left after the CIC removes the sum term is
#     I = A*L/2^7 = 2^21, since cos*cos = 0.5cos(diff) + 0.5cos(sum)
#   CIC output: same scale as its 24-bit input, plus 8 bits   -> x 2^8
#                                                (red_pitaya_acq_cic.v)
#
#   => |z| for a full-scale sine at the converter = 2^21 * 2^8 = 2^29
#
# It does not move with decimation: the CIC's output slice tracks R exactly so
# that the scale stays put. That is the whole reason R is a power of two.
#
# The FIR droop corrector downstream has a DC gain of 1.02 to 1.3 depending on
# its corner, so the true clip point is up to 2.3 dB below this. That is inside
# the 6 dB margin MIN_HEADROOM_DB already asks for.
FULL_SCALE      = float(1 << 29)              # CIC output, see derivation above
ADC_FULL_SCALE  = float(1 << 13)              # the converter itself, for reference
MIN_HEADROOM_DB = 6.0
DEC_CHK         = 10

def level_report(f_lo, dec=DEC_CHK, n=8192):
    """Headroom and peak-to-average ratio for one record."""
    z = capture(f_lo, dec, n)
    mag = np.abs(z)
    rms = float(np.sqrt(np.mean(mag ** 2)))
    pk = float(np.max(mag))
    near_full = int(np.count_nonzero(mag > 0.98 * FULL_SCALE))
    headroom = 20.0 * np.log10(FULL_SCALE / max(pk, 1e-9))
    crest = pk / max(rms, 1e-30)
    expected = float(np.sqrt(np.log(len(z))))   # what pure noise gives
    return dict(rms=rms, peak=pk, near_full=near_full, headroom=headroom,
                crest=crest, expected=expected)


print('source: %s' % SOURCE_NOTE)
print()
r = level_report(F0)
print('  rms                    %.4e counts' % r['rms'])
print('  largest sample         %.4e counts' % r['peak'])
print('  headroom to full scale %.1f dB' % r['headroom'])
print('  samples near full scale %d of 8192' % r['near_full'])
print('  peak/average           %.2f   (pure noise gives about %.2f)'
      % (r['crest'], r['expected']))
print()

ok_head = r['headroom'] > MIN_HEADROOM_DB and r['near_full'] == 0
ok_crest = r['crest'] > 0.6 * r['expected']
if not ok_head:
    print('  TOO LOUD - reduce the gain ahead of the converter before measuring.')
if not ok_crest:
    print('  The peaks are missing: the signal is being squashed even though')
    print('  little or nothing has reached the rail. Reduce the gain.')
print('STEP 1:', 'PASS' if (ok_head and ok_crest) else 'FAIL - the input is not linear')

---
## Step 2 — What else is in the band?

Digital boards are full of clock harmonics and power-supply whistles. One sitting near the coil
frequency would be picked up by Stage 15's search and fitted as a resonance with an
extraordinary Q — and nothing downstream would contradict it.

So sweep the whole usable band with the input terminated and catalogue every line that stands
above the floor. This list is the thing to consult when a later measurement finds something
surprising.

In [ ]:
SURVEY_LO, SURVEY_HI = 1.0e6, 60.0e6      # the usable band
SURVEY_STEP          = 0.5e6
SPUR_THRESHOLD_DB    = 10.0               # above the median floor

print('sweeping %.0f to %.0f MHz in %.1f MHz steps, input terminated'
      % (SURVEY_LO / 1e6, SURVEY_HI / 1e6, SURVEY_STEP / 1e6))
fs_survey = np.arange(SURVEY_LO, SURVEY_HI + 1.0, SURVEY_STEP)
pwr = np.array([float(np.var(capture(f, 10, 2048))) for f in fs_survey])

floor = float(np.median(pwr))
db = 10.0 * np.log10(np.maximum(pwr, 1e-30) / max(floor, 1e-30))
spurs = [(f, d) for f, d in zip(fs_survey, db) if d > SPUR_THRESHOLD_DB]

print()
print('median floor: %.4e' % floor)
if spurs:
    print('%d line(s) more than %.0f dB above it:' % (len(spurs), SPUR_THRESHOLD_DB))
    for f, d in spurs:
        print('   %10.3f MHz   %+6.1f dB' % (f / 1e6, d))
else:
    print('nothing more than %.0f dB above the floor' % SPUR_THRESHOLD_DB)

plt.close('all')
plt.figure(figsize=(11, 3.6))
plt.plot(fs_survey / 1e6, db, '-', lw=1)
plt.axhline(SPUR_THRESHOLD_DB, color='r', ls=':', lw=1, label='threshold')
plt.xlabel('frequency [MHz]'); plt.ylabel('dB above the median floor')
plt.title('What is in the band, with nothing connected')
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

# Anything close to where the coil is expected is the dangerous case.
COIL_EXPECTED = F0
near = [(f, d) for f, d in spurs if abs(f - COIL_EXPECTED) < 2.0e6]
print()
if near:
    print('WARNING: %d line(s) within 2 MHz of the expected coil frequency:' % len(near))
    for f, d in near:
        print('   %10.3f MHz   %+6.1f dB  <- Stage 15 could fit this as a resonance' % (f / 1e6, d))
    print('   Move the operating frequency, or note these before searching.')
else:
    print('nothing within 2 MHz of %.3f MHz - the search band is clear' % (COIL_EXPECTED / 1e6))
print()
print('STEP 2: survey complete (a catalogue, not a pass or fail)')

---
## Step 3 — Do the three measurement paths agree?

There are three ways to measure the same signal here, each with its own scaling:

- the **register voltmeter**, which averages the receiver output directly
- the **memory capture**, which records a time series
- the **averaged spectrum**, built from that time series

Stage 7 already checks the memory capture against a known signal. Nothing checks the other two
against it. A scaling error confined to one path — a stray factor of two, a window not undone —
would be invisible today, and would show up later as a physics result that disagrees with
itself.

One steady tone, measured three ways, converted to the same units.

In [ ]:
# ============================================================================
#  STEP 3 - do the three measurement paths agree?
#
#  WHAT WAS WRONG WITH THIS STEP  (fixed 2026-08-03)
#
#  It started the generator and compared the paths - but nothing the generator
#  played ever reached the converter, so all three were measuring the noise
#  floor of a terminated input. Two faults, and both had to be fixed.
#
#  1. NO SIGNAL PATH. The top level builds the converter word as
#
#         adc_dat[0]     = digital_loop ? dac_a : <the real converter>
#         adc_dat_ps[0]  = reso_ena     ? reso_dat : adc_dat[0]
#
#     and the acquisition taps adc_dat_ps. With the bench input terminated,
#     digital_loop off and the resonator disabled - which is how this step left
#     things - the generator played into the DAC and the acquisition looked at
#     a terminated input. The tone existed; it simply had nowhere to go.
#
#     The internal loopback is the right source for a scaling cross-check:
#     unity gain, no cable, no resonance, no analogue, nothing to calibrate.
#
#  2. THE PATHS WERE NOT COMPARABLE. The register voltmeter returns
#     |mean(z)| - a COHERENT average, which cancels anything not at the tuned
#     frequency. The other two returned sqrt(mean(|z|^2)) - TOTAL POWER, which
#     cancels nothing. On a clean on-tune tone these agree; on noise they differ
#     by the square root of the number of samples averaged, which is exactly the
#     factor 7.2 this step reported.
#
#     Worse, the "third path" was not independent: by Parseval, the spectrum
#     figure IS the record's mean square computed a second way. It agreed with
#     the capture to 1.1% because it could hardly do otherwise. Two paths were
#     being presented as three.
#
#     So compare like with like: coherent against coherent, power against power.
# ============================================================================
hk = Bus(0x40000000)                 # house keeping - holds the loopback switch
loop_saved = hk.r(0x0C) & 1
hk.w(0x0C, 1)                        # dac_a -> adc_dat[0], straight through

try:
    gen.load_sine(); gen.freq(F0); gen.amp(0.15); gen.run()
    acq.start(dec_log2=DEC_CHK); acq.tune(F0)
    time.sleep(0.2)

    # 1. the register voltmeter - a coherent average
    a_peek_coh = abs(acq.measure(256))

    # 2. the memory capture, reduced BOTH ways from the same record
    z = capture(F0, DEC_CHK, 8192)
    a_cap_coh = float(abs(np.mean(z)))                        # coherent
    a_cap_rms = float(np.sqrt(np.mean(np.abs(z) ** 2)))       # total power

    # 3. the averaged spectrum - undo the window and the transform length, then
    #    take the power in the line rather than its height, so the comparison
    #    does not depend on where the tone sits inside a bin.
    f_psd, p_psd = welch_psd(z, seg=2048)
    w_seg = np.hanning(2048)
    a_spec_rms = float(np.sqrt(np.sum(p_psd) / (2048.0 ** 2 * np.mean(w_seg ** 2))))
finally:
    gen.amp(0.0)
    hk.w(0x0C, loop_saved)           # leave the input as it was found

# How much of the record is the tone. Near 1 means a clean tone; near 0 means
# there is nothing here but noise, and no ratio below is worth reading.
coh_frac = a_cap_coh / max(a_cap_rms, 1e-30)

print('%-34s %14s' % ('coherent amplitude  |mean z|', ''))
print('  %-32s %14.4e' % ('register voltmeter', a_peek_coh))
print('  %-32s %14.4e' % ('memory capture', a_cap_coh))
print('  ratio                            %14.3f' % (a_peek_coh / max(a_cap_coh, 1e-30)))
print()
print('%-34s %14s' % ('total power  sqrt(mean |z|^2)', ''))
print('  %-32s %14.4e' % ('memory capture', a_cap_rms))
print('  %-32s %14.4e' % ('averaged spectrum', a_spec_rms))
print('  ratio                            %14.3f' % (a_spec_rms / max(a_cap_rms, 1e-30)))
print()
print('tone fraction of the record: %.3f' % coh_frac)

ok_tone  = coh_frac > 0.7
ok_coh   = abs(a_peek_coh / max(a_cap_coh, 1e-30) - 1.0) < 0.15
ok_power = abs(a_spec_rms / max(a_cap_rms, 1e-30) - 1.0) < 0.15

if not ok_tone:
    print()
    print('  NO TONE. The record is mostly noise, so the paths below are being')
    print('  asked to agree about something that is not there. Check that the')
    print('  loopback took (house keeping 0x0C bit 0), that the generator is')
    print('  running, and that the acquisition is tuned to the same frequency.')
elif not ok_coh:
    print()
    print('  The two COHERENT paths disagree by more than 15%. Both reduce the')
    print('  same stream the same way, so this is a scaling error between the')
    print('  peek registers and the memory writer - not a signal problem.')
elif not ok_power:
    print()
    print('  The two POWER paths disagree by more than 15%. Since both come from')
    print('  one record, this is an error in the spectrum normalisation, not in')
    print('  the instrument.')

print('STEP 3:', 'PASS' if (ok_tone and ok_coh and ok_power) else
      'FAIL - the measurement paths disagree')


---
## Stage 13 summary

| Step | What it delivered |
|---|---|
| 1 | Headroom and peak-to-average ratio, and a refusal to proceed when the input is being squashed |
| 2 | A catalogue of everything in the band, and a warning if any of it sits near where the coil is expected |
| 3 | Agreement between the three ways this system can measure the same signal |

Run it again with the coil and amplifier attached: Step 1 is the one that changes, and it is
the one that matters most, because nothing else in the project will notice an overloaded front
end.

In [ ]:
gen.stop(); ps.off(); acq.w(0x00, 0)
print('all quiet')